# 罗伯·瑞克超额现金流选股法则 - 优化版

## 策略概述

- 罗伯·瑞克，华尔街著名的资本大鳄，从事证券行业近 50 年。本篇介绍的罗伯·瑞克超额现金流选股法则是由其在接受股票书籍作者采访时，给投资者的投资建议所提炼而成。
- 该策略从三个角度对投资标的进行考察：估值水平、分红水平、财务状况
- 使用免费数据源：tushare + akshare，无需付费数据
- 代码优化：使用现代化Python库，提高代码可读性和执行效率

## 选股标准（中国化改造版本）

1. 市净率低于 3
2. 股利收益率高于市场平均值 
3. 市盈率低于市场平均值
4. 借款总额占总资本比例低于 33%
5. 股价/超额现金流量比低于市场平均的80%

In [ ]:
# 导入必要的库
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import tushare as ts
import akshare as ak
from datetime import datetime, timedelta
import time
from typing import Dict, List, Optional, Tuple
import logging

# 可视化库
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8')

# 设置pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

print("库导入完成")

In [ ]:
# 配置tushare
TUSHARE_TOKEN = "ce9f8c37a4af987f6328321ed4b3f9379f695d6d6bdc5b59d454e3ad"
ts.set_token(TUSHARE_TOKEN)
pro = ts.pro_api()

print("Tushare配置完成")

In [ ]:
class DataProvider:
    """数据提供者类，封装所有数据获取逻辑"""
    
    def __init__(self, token: str):
        self.pro = ts.pro_api(token)
        self.logger = self._setup_logger()
        
    def _setup_logger(self) -> logging.Logger:
        """设置日志"""
        logger = logging.getLogger('DataProvider')
        logger.setLevel(logging.INFO)
        if not logger.handlers:
            handler = logging.StreamHandler()
            formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            handler.setFormatter(formatter)
            logger.addHandler(handler)
        return logger
    
    def get_stock_list(self, trade_date: str) -> pd.DataFrame:
        """获取股票列表，过滤ST、新股等"""
        try:
            # 获取基础股票信息
            stocks = self.pro.stock_basic(
                exchange='', 
                list_status='L',
                fields='ts_code,symbol,name,area,industry,list_date'
            )
            
            # 过滤条件
            # 1. 过滤ST股票
            stocks = stocks[~stocks['name'].str.contains('ST|退', na=False)]
            
            # 2. 过滤上市不足180天的新股
            trade_dt = pd.to_datetime(trade_date)
            stocks['list_date'] = pd.to_datetime(stocks['list_date'])
            stocks = stocks[stocks['list_date'] <= trade_dt - timedelta(days=180)]
            
            # 3. 过滤银行股
            stocks = stocks[stocks['industry'] != '银行']
            
            self.logger.info(f"获取到{len(stocks)}只股票")
            return stocks
            
        except Exception as e:
            self.logger.error(f"获取股票列表失败: {e}")
            return pd.DataFrame()
    
    def get_daily_basic(self, trade_date: str, ts_codes: List[str] = None) -> pd.DataFrame:
        """获取每日基本面数据"""
        try:
            # 分批获取数据，避免接口限制
            all_data = []
            
            if ts_codes:
                # 分批处理
                batch_size = 1000
                for i in range(0, len(ts_codes), batch_size):
                    batch_codes = ts_codes[i:i+batch_size]
                    batch_data = self.pro.daily_basic(
                        trade_date=trade_date.replace('-', ''),
                        ts_code=','.join(batch_codes),
                        fields='ts_code,trade_date,pe_ttm,pb,dv_ttm,total_mv'
                    )
                    all_data.append(batch_data)
                    time.sleep(0.1)  # 避免频率限制
            else:
                # 获取全市场数据
                all_data = [self.pro.daily_basic(
                    trade_date=trade_date.replace('-', ''),
                    fields='ts_code,trade_date,pe_ttm,pb,dv_ttm,total_mv'
                )]
            
            result = pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()
            self.logger.info(f"获取到{len(result)}条每日基本面数据")
            return result
            
        except Exception as e:
            self.logger.error(f"获取每日基本面数据失败: {e}")
            return pd.DataFrame()
    
    def get_financial_data(self, period: str, ts_codes: List[str] = None) -> Dict[str, pd.DataFrame]:
        """获取财务数据"""
        try:
            result = {}
            
            # 获取资产负债表数据
            balance_data = self._get_balance_sheet(period, ts_codes)
            result['balance'] = balance_data
            
            # 获取现金流量表数据
            cashflow_data = self._get_cashflow(period, ts_codes)
            result['cashflow'] = cashflow_data
            
            return result
            
        except Exception as e:
            self.logger.error(f"获取财务数据失败: {e}")
            return {}
    
    def _get_balance_sheet(self, period: str, ts_codes: List[str] = None) -> pd.DataFrame:
        """获取资产负债表数据"""
        all_data = []
        
        if ts_codes:
            batch_size = 50  # 财务数据接口限制更严格
            for i in range(0, len(ts_codes), batch_size):
                batch_codes = ts_codes[i:i+batch_size]
                for code in batch_codes:
                    try:
                        data = self.pro.balancesheet(
                            ts_code=code,
                            period=period,
                            fields='ts_code,ann_date,end_date,total_assets,total_liab,lt_borr,st_borr'
                        )
                        if not data.empty:
                            all_data.append(data)
                        time.sleep(0.2)  # 财务数据需要更长间隔
                    except Exception as e:
                        self.logger.warning(f"获取{code}资产负债表失败: {e}")
                        continue
        
        return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()
    
    def _get_cashflow(self, period: str, ts_codes: List[str] = None) -> pd.DataFrame:
        """获取现金流量表数据"""
        all_data = []
        
        if ts_codes:
            batch_size = 50
            for i in range(0, len(ts_codes), batch_size):
                batch_codes = ts_codes[i:i+batch_size]
                for code in batch_codes:
                    try:
                        data = self.pro.cashflow(
                            ts_code=code,
                            period=period,
                            fields='ts_code,ann_date,end_date,n_cashflow_act'
                        )
                        if not data.empty:
                            all_data.append(data)
                        time.sleep(0.2)
                    except Exception as e:
                        self.logger.warning(f"获取{code}现金流量表失败: {e}")
                        continue
        
        return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

# 初始化数据提供者
data_provider = DataProvider(TUSHARE_TOKEN)
print("数据提供者初始化完成")

In [ ]:
class RobertRickStrategy:
    """罗伯·瑞克超额现金流选股策略"""
    
    def __init__(self, data_provider: DataProvider):
        self.data_provider = data_provider
        self.logger = self._setup_logger()
        
    def _setup_logger(self) -> logging.Logger:
        """设置日志"""
        logger = logging.getLogger('RobertRickStrategy')
        logger.setLevel(logging.INFO)
        if not logger.handlers:
            handler = logging.StreamHandler()
            formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            handler.setFormatter(formatter)
            logger.addHandler(handler)
        return logger
    
    def screen_stocks(self, trade_date: str, period: str) -> pd.DataFrame:
        """执行选股策略"""
        self.logger.info(f"开始执行选股策略，交易日期: {trade_date}, 财报期间: {period}")
        
        # 1. 获取股票池
        stocks = self.data_provider.get_stock_list(trade_date)
        if stocks.empty:
            self.logger.error("未获取到股票数据")
            return pd.DataFrame()
        
        # 2. 获取每日基本面数据
        daily_basic = self.data_provider.get_daily_basic(trade_date, stocks['ts_code'].tolist())
        if daily_basic.empty:
            self.logger.error("未获取到每日基本面数据")
            return pd.DataFrame()
        
        # 3. 获取财务数据
        financial_data = self.data_provider.get_financial_data(period, stocks['ts_code'].tolist())
        
        # 4. 合并数据
        merged_data = self._merge_data(stocks, daily_basic, financial_data)
        
        # 5. 应用筛选条件
        selected_stocks = self._apply_filters(merged_data)
        
        self.logger.info(f"筛选完成，共选出{len(selected_stocks)}只股票")
        return selected_stocks
    
    def _merge_data(self, stocks: pd.DataFrame, daily_basic: pd.DataFrame, 
                   financial_data: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        """合并各类数据"""
        # 合并基础信息和每日数据
        merged = pd.merge(stocks, daily_basic, on='ts_code', how='left')
        
        # 合并财务数据
        if 'balance' in financial_data and not financial_data['balance'].empty:
            balance = financial_data['balance'].groupby('ts_code').first().reset_index()
            merged = pd.merge(merged, balance, on='ts_code', how='left')
        
        if 'cashflow' in financial_data and not financial_data['cashflow'].empty:
            cashflow = financial_data['cashflow'].groupby('ts_code').first().reset_index()
            merged = pd.merge(merged, cashflow, on='ts_code', how='left')
        
        # 计算衍生指标
        merged = self._calculate_derived_metrics(merged)
        
        return merged
    
    def _calculate_derived_metrics(self, df: pd.DataFrame) -> pd.DataFrame:
        """计算衍生指标"""
        # 借款总额占总资产比例
        df['lt_borr'] = df['lt_borr'].fillna(0)
        df['st_borr'] = df['st_borr'].fillna(0)
        df['debt_ratio'] = (df['lt_borr'] + df['st_borr']) / df['total_assets']
        
        # 股价现金流比率（使用经营活动现金流净额代替自由现金流）
        df['price_cashflow_ratio'] = df['total_mv'] / (df['n_cashflow_act'] / 10000)  # 转换单位
        
        return df
    
    def _apply_filters(self, df: pd.DataFrame) -> pd.DataFrame:
        """应用筛选条件"""
        # 移除缺失关键数据的股票
        df = df.dropna(subset=['pe_ttm', 'pb', 'dv_ttm', 'debt_ratio', 'price_cashflow_ratio'])
        
        if df.empty:
            self.logger.warning("数据清洗后无可用股票")
            return df
        
        # 计算市场平均值
        market_avg_pe = df['pe_ttm'].mean()
        market_avg_div = df['dv_ttm'].mean()
        market_avg_pcf = df['price_cashflow_ratio'].mean() * 0.8
        
        self.logger.info(f"市场平均PE: {market_avg_pe:.2f}")
        self.logger.info(f"市场平均股息率: {market_avg_div:.2f}%")
        self.logger.info(f"市场平均价现比80%: {market_avg_pcf:.2f}")
        
        # 应用筛选条件
        conditions = (
            (df['pb'] < 3) &                              # 1. 市净率低于3
            (df['dv_ttm'] > market_avg_div) &             # 2. 股息率高于市场平均
            (df['pe_ttm'] < market_avg_pe) &              # 3. 市盈率低于市场平均
            (df['debt_ratio'] < 0.33) &                   # 4. 负债率低于33%
            (df['price_cashflow_ratio'] < market_avg_pcf) # 5. 价现比低于市场平均80%
        )
        
        selected = df[conditions].copy()
        
        # 添加筛选统计信息
        selected['market_avg_pe'] = market_avg_pe
        selected['market_avg_div'] = market_avg_div
        selected['market_avg_pcf_80'] = market_avg_pcf
        
        return selected

# 初始化策略
strategy = RobertRickStrategy(data_provider)
print("策略初始化完成")

In [ ]:
# 执行选股策略示例
def run_strategy_example():
    """运行策略示例"""
    # 设置参数
    trade_date = '2023-12-29'  # 交易日期
    period = '20230930'        # 财报期间（2023年三季报）
    
    print(f"开始执行罗伯·瑞克选股策略...")
    print(f"交易日期: {trade_date}")
    print(f"财报期间: {period}")
    print("-" * 50)
    
    # 执行选股
    selected_stocks = strategy.screen_stocks(trade_date, period)
    
    if not selected_stocks.empty:
        print(f"\n选股结果：")
        print(f"共筛选出 {len(selected_stocks)} 只股票")
        
        # 显示前10只股票的详细信息
        display_columns = [
            'ts_code', 'name', 'pe_ttm', 'pb', 'dv_ttm', 
            'debt_ratio', 'price_cashflow_ratio'
        ]
        
        print("\n前10只股票详情：")
        print(selected_stocks[display_columns].head(10).to_string(index=False))
        
        # 统计信息
        print("\n统计信息：")
        print(f"平均PE: {selected_stocks['pe_ttm'].mean():.2f}")
        print(f"平均PB: {selected_stocks['pb'].mean():.2f}")
        print(f"平均股息率: {selected_stocks['dv_ttm'].mean():.2f}%")
        print(f"平均负债率: {selected_stocks['debt_ratio'].mean():.2f}")
        
        return selected_stocks
    else:
        print("未找到符合条件的股票")
        return pd.DataFrame()

# 注意：由于数据获取需要时间，建议分批运行
print("策略准备就绪，可以调用 run_strategy_example() 执行选股")

In [ ]:
# 可视化分析工具
class StrategyAnalyzer:
    """策略分析工具"""
    
    @staticmethod
    def plot_distribution(df: pd.DataFrame, column: str, title: str = None):
        """绘制指标分布图"""
        if df.empty or column not in df.columns:
            print(f"数据为空或不包含列: {column}")
            return
        
        plt.figure(figsize=(10, 6))
        plt.hist(df[column].dropna(), bins=30, alpha=0.7, edgecolor='black')
        plt.title(title or f'{column} 分布图')
        plt.xlabel(column)
        plt.ylabel('频数')
        plt.grid(True, alpha=0.3)
        plt.show()
    
    @staticmethod
    def plot_scatter(df: pd.DataFrame, x_col: str, y_col: str, title: str = None):
        """绘制散点图"""
        if df.empty or x_col not in df.columns or y_col not in df.columns:
            print(f"数据为空或不包含必要列")
            return
        
        plt.figure(figsize=(10, 6))
        plt.scatter(df[x_col], df[y_col], alpha=0.6)
        plt.title(title or f'{x_col} vs {y_col}')
        plt.xlabel(x_col)
        plt.ylabel(y_col)
        plt.grid(True, alpha=0.3)
        plt.show()
    
    @staticmethod
    def generate_report(df: pd.DataFrame) -> str:
        """生成分析报告"""
        if df.empty:
            return "无数据可分析"
        
        report = []
        report.append("=" * 50)
        report.append("罗伯·瑞克选股策略分析报告")
        report.append("=" * 50)
        report.append(f"选中股票数量: {len(df)}")
        report.append("")
        
        # 各指标统计
        metrics = ['pe_ttm', 'pb', 'dv_ttm', 'debt_ratio', 'price_cashflow_ratio']
        for metric in metrics:
            if metric in df.columns:
                data = df[metric].dropna()
                if not data.empty:
                    report.append(f"{metric}:")
                    report.append(f"  平均值: {data.mean():.4f}")
                    report.append(f"  中位数: {data.median():.4f}")
                    report.append(f"  标准差: {data.std():.4f}")
                    report.append(f"  最小值: {data.min():.4f}")
                    report.append(f"  最大值: {data.max():.4f}")
                    report.append("")
        
        return "\n".join(report)

# 初始化分析器
analyzer = StrategyAnalyzer()
print("分析工具初始化完成")

## 使用说明

### 1. 执行选股策略
```python
# 运行选股策略
result = run_strategy_example()
```

### 2. 分析结果
```python
# 生成分析报告
report = analyzer.generate_report(result)
print(report)

# 绘制指标分布图
analyzer.plot_distribution(result, 'pe_ttm', 'PE分布图')
analyzer.plot_distribution(result, 'pb', 'PB分布图')

# 绘制散点图
analyzer.plot_scatter(result, 'pe_ttm', 'pb', 'PE vs PB')
```

### 3. 自定义参数
```python
# 自定义日期和期间
custom_result = strategy.screen_stocks('2023-06-30', '20230331')
```

## 优化特点

1. **数据源替换**: 完全使用tushare免费接口，无需jqdata
2. **代码结构**: 采用面向对象设计，代码更清晰易维护
3. **错误处理**: 添加完善的异常处理和日志记录
4. **性能优化**: 分批获取数据，避免接口限制
5. **可视化**: 提供丰富的分析和可视化工具
6. **文档完善**: 详细的使用说明和代码注释

## 注意事项

1. 首次运行需要较长时间获取数据
2. 建议使用tushare积分用户以获得更好的接口体验
3. 财务数据存在发布延迟，建议使用已发布的财报期间
4. 策略结果仅供参考，投资需谨慎